Imports and Dataset Download

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import kagglehub
import json

# 1. Download Dataset
print("Downloading CASIA 2.0 via kagglehub...")
dataset_download_path = kagglehub.dataset_download("divg07/casia-20-image-tampering-detection-dataset")

# 2. Robust Folder Discovery
authentic_images = []
forged_images = []

for root, dirs, files in os.walk(dataset_download_path):
    if os.path.basename(root) == 'Au':
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.tif', '.bmp')):
                authentic_images.append(os.path.join(root, f))
    if os.path.basename(root) == 'Tp':
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.tif', '.bmp')):
                forged_images.append(os.path.join(root, f))

print(f"✅ Found {len(authentic_images)} Authentic and {len(forged_images)} Forged images.")

Data Splitting and Pipeline

In [ ]:
# 1. Combine and Label
all_image_paths = authentic_images + forged_images
all_labels = np.array([0] * len(authentic_images) + [1] * len(forged_images), dtype=np.int32)

# 2. Train/Val/Test Split
X_train_paths, X_temp_paths, y_train, y_temp = train_test_split(
    all_image_paths, all_labels, test_size=0.3, stratify=all_labels, random_state=42
)
X_val_paths, X_test_paths, y_val, y_test = train_test_split(
    X_temp_paths, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

# 3. TF Data Pipeline
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
    return img, label

train_ds = tf.data.Dataset.from_tensor_slices((X_train_paths, y_train))\
    .shuffle(1000).map(load_and_preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val_paths, y_val))\
    .map(load_and_preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((X_test_paths, y_test))\
    .map(load_and_preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

Model Building and Fine-Tuning

In [ ]:
# 1. Base Model
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights='imagenet'
)

# 2. Unfreeze top 30 layers
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

# 3. Add Custom Head
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),  # High dropout to prevent the 97%/50% accuracy gap
    layers.Dense(1, activation='sigmoid')
])

# 4. Compile with Low Learning Rate
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Starting Fine-Tuning...")
history = model.fit(train_ds, validation_data=val_ds, epochs=10)

Evaluation and Saving

In [ ]:
# 1. Evaluate
results = model.evaluate(test_ds)
y_pred_probs = model.predict(test_ds)
y_pred = (y_pred_probs > 0.5).astype(int)

# Since test_ds order might differ from y_test due to batching, 
# we grab labels directly from the dataset for the matrix
true_labels = np.concatenate([y for x, y in test_ds], axis=0)
pred_labels = (model.predict(test_ds) > 0.5).astype(int)

print("\n--- Confusion Matrix ---")
print(confusion_matrix(true_labels, pred_labels))
print("\n--- Classification Report ---")
print(classification_report(true_labels, pred_labels))

# 2. Save Artifacts
model.save("mobilenet_casia_finetuned.h5")
metadata = {
    "model": "MobileNetV2-FineTuned",
    "test_accuracy": float(results[1]),
    "train_samples": len(X_train_paths)
}
with open("metadata.json", "w") as f:
    json.dump(metadata, f)

# 3. Download
from google.colab import files
files.download("mobilenet_casia_finetuned.h5")
files.download("metadata.json")